In [2]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: d:\Programing\CyberSec-Reasoner


In [ ]:
import torch
import random
import logging
import warnings

%load_ext autoreload
%autoreload 2

from datasets import DatasetDict
from src.utils.logger import setup_logging
from src.config_loader import load_config
from src.utils.utils import setup_env, log_gpu_info
from src.utils.seed import setup_seed
from src.utils.wandb import init_wandb, finish_wandb
from src.model_loader import load_tokenizer, load_qlora_base_model
from src.data_loader import load_dataset, format_for_grpo
from src.dataset_stats import check_answer_length
from src.reward_functions import hybrid_reward
from src.trainer import build_grpo_config, build_grpo_trainer, save_adapter, merge_and_save

warnings.filterwarnings("ignore")
logger = logging.getLogger(__name__)

In [ ]:
if torch.cuda.is_available():
    print(f"Number of available GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties().total_memory / 1024**3:.2f} GB")
else:
    print("No GPU detected! Using CPU...")

*****
# Training

In [ ]:
## setup logging
setup_logging()

config_path = "./src/configs/grpo_qwen3.5_4b.yaml"
model_cfg, dataset_cfg, lora_config, training_cfg, wandb_cfg, paths_cfg, _ = load_config(config_path)

# setup environment
setup_env()

# log GPU info
log_gpu_info()

# setup seed
setup_seed(training_cfg)

# init wandb
wandb_run = init_wandb(training_cfg["report_to"], wandb_cfg)

In [ ]:
# tokenizer
tokenizer = load_tokenizer(model_cfg, dataset_cfg)

In [ ]:
# dataset
dataset = load_dataset(dataset_cfg)

# take 10 of the dataset for cold-start reasoining sft
cold_start = DatasetDict({
    "train": dataset["train"].select(range(10)).shuffle(seed=42)
})

dataset = cold_start

# formatt the dataset for grpo training
dataset = dataset["train"].map(
    format_for_grpo,
    remove_columns=dataset["train"].column_names,
)

# check answer lengths
check_answer_length(dataset, tokenizer)

In [ ]:
print(dataset[random.randint(0, len(dataset)-1)])

In [ ]:
# load model
log_gpu_info("Before loading model")
model = load_qlora_base_model(model_cfg)
log_gpu_info("After loading model")

In [ ]:
# init trainer
grpo_config = build_grpo_config(training_cfg, wandb_cfg["run_name"])

trainer = build_grpo_trainer(
    model, 
    tokenizer, 
    dataset,
    grpo_config,
    lora_config,
    hybrid_reward
)

In [ ]:
# start training
log_gpu_info("Before training")
logger.info("Starting training...")
result = trainer.train()
log_gpu_info("After training")
logger.info(f"Training completed. Training result: {result}")

In [43]:
ds['train'][0]['messages'][0]

{'content': 'You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.\n\nYour task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).\n\nYou must follow a structured reasoning workflow:\n\n1. Understand the vulnerability context and affected components  \n2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  \n3. Determine the root cause of the weakness  \n4. Map the root cause to the most appropriate CWE category  \n\nGuidelines:\n- Focus on root cause rather than surface-level keywords  \n- Use precise cybersecurity terminology  \n- Ensure reasoning clearly supports the final CWE selection  \n- Prefer the most specific applicable CWE when possible  \n\nAvoid:\n- Guessing without justification  \n- Contradictions between reasoning and conclusion  \n- Irrelevant or overly generic explanations  \n- Blindly copying CWE iden

In [ ]:
# save
save_adapter(trainer, tokenizer, paths_cfg)
merge_and_save(model_cfg, paths_cfg)

In [ ]:
finish_wandb(
    wandb_run
)
logger.info("Wandb run finished.")